# Combine linear regression with Random Forest + Extra Trees
Simplified, self-contained version of `ensemble2.ipynb`. Run top to bottom in `amazon_ml` from the project folder.

**Intentional model change:** the old notebook imports a missing `final_dt_submission.ipynb`, containing an additional bagging branch and EY specialist. This version uses the available RF + Extra Trees model from `flight_dt.ipynb`, as requested. Old ensemble scores (including public R² 0.96154) must not be attributed to this version. Run the evaluation below to measure its performance.

The LR pipeline and ensemble methods are retained. Model definitions are included here, so no notebook loader or missing notebook is required.

## 1. Load data and libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
DATA_DIR = Path('data')

def score(y, prediction):
    return {
        'RMSE': np.sqrt(mean_squared_error(y, prediction)),
        'MAE': mean_absolute_error(y, prediction),
        'R2': r2_score(y, prediction),
    }

def write_submission(test, prediction, filename):
    prediction = np.asarray(prediction)
    assert prediction.shape == (len(test),)
    assert np.isfinite(prediction).all() and (prediction > 0).all()
    result = pd.DataFrame({'id': test['id'].to_numpy(), 'price': prediction})
    assert result['id'].is_unique
    result.to_csv(DATA_DIR / filename, index=False)
    print('Saved:', DATA_DIR / filename)
    return result

In [2]:
from collections import defaultdict
from scipy.optimize import nnls
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
print('Train:', train.shape, '| Test:', test.shape)
display(pd.DataFrame({'train_missing': train.isna().sum(), 'test_missing': test.isna().sum()}))

Train: (15589, 12) | Test: (3898, 11)


,train_missing,test_missing
airline,3089,799.0
arrival_time,0,0.0
days_left,0,0.0
departure_time,0,0.0
destination_city,0,0.0
duration,3095,776.0
flight,0,0.0
flight_class,0,0.0
id,0,0.0
price,0,NaN


## 2. Define the RF + Extra Trees model
Same original features, target encoding, log-price blend, and five-fold averaging as the simplified DT notebook.

In [4]:
def original_features(train, test):
    a = train.copy()
    b = test.copy()
    reference = pd.concat([a, b], ignore_index=True)

    reference["flight_prefix"] = (
        reference["flight"].str.split().str[0]
    )

    prefix_map = (
        reference.dropna(subset=["airline"])
        .drop_duplicates("flight_prefix")
        .set_index("flight_prefix")["airline"].to_dict()
    )

    for df in [a, b]:
        df["flight_prefix"] = df["flight"].str.split().str[0]
        df["airline"] = df["airline"].fillna(
            df["flight_prefix"].map(prefix_map)
        )
        df["route"] = (
            df["source_city"] + "_" + df["destination_city"]
        )
        df["airline_class"] = (
            df["airline"] + "_" + df["flight_class"]
        )
        df["route_class"] = (
            df["route"] + "_" + df["flight_class"]
        )
        df["airline_route"] = (
            df["airline"] + "_" + df["route"]
        )
        df["route_stops"] = (
            df["route"] + "_" + df["stops"]
        )

    flight_route_duration = reference.groupby(
        ["flight", "source_city", "destination_city"]
    )["duration"].median().to_dict()

    flight_duration = (
        reference.groupby("flight")["duration"].median().to_dict()
    )
    route_stops_duration = a.groupby(
        ["route", "stops"]
    )["duration"].median().to_dict()

    global_duration = a["duration"].median()

    def recover_duration(row):
        if pd.notna(row["duration"]):
            return row["duration"]

        value = flight_route_duration.get((
            row["flight"], row["source_city"], row["destination_city"]
        ))
        if pd.notna(value):
            return value

        value = flight_duration.get(row["flight"])
        if pd.notna(value):
            return value

        return route_stops_duration.get(
            (row["route"], row["stops"]), global_duration
        )

    for df in [a, b]:
        df["duration"] = df.apply(recover_duration, axis=1)

    minimum_duration = a.groupby("route")["duration"].min().to_dict()

    time_map = {
        "Early_Morning": 0, "Morning": 1, "Afternoon": 2,
        "Evening": 3, "Night": 4, "Late_Night": 5,
    }

    for df in [a, b]:
        df["min_route_dur"] = (
            df["route"].map(minimum_duration).fillna(1.5)
        )
        df["layover_est"] = (
            df["duration"] - df["min_route_dur"]
        ).clip(lower=0)

        df["flight_class_num"] = df["flight_class"].map({
            "Economy": 0, "Business": 1
        })
        df["stops_num"] = df["stops"].map({
            "zero": 0, "one": 1, "two_or_more": 2
        })
        df["dep_time_num"] = (
            df["departure_time"].map(time_map).fillna(-1)
        )
        df["arr_time_num"] = (
            df["arrival_time"].map(time_map).fillna(-1)
        )

        df["is_direct"] = (df["stops_num"] == 0).astype(int)
        df["is_last_minute"] = (df["days_left"] <= 2).astype(int)
        df["is_urgent"] = (df["days_left"] <= 5).astype(int)
        df["is_early_bird"] = (df["days_left"] >= 25).astype(int)
        df["days_left_inv"] = 1 / (df["days_left"] + 1)
        df["days_left_log"] = np.log1p(df["days_left"])
        df["duration_log"] = np.log1p(df["duration"])
        df["duration_per_stop"] = (
            df["duration"] / (df["stops_num"] + 1)
        )
        df["speed_proxy"] = (
            df["min_route_dur"] / (df["duration"] + 0.1)
        )

    for column in ["flight", "airline", "route", "airline_route"]:
        counts = a[column].value_counts().to_dict()
        for df in [a, b]:
            df[f"{column}_freq"] = (
                df[column].map(counts).fillna(0)
            )

    categorical = [
        "flight_prefix", "airline", "source_city", "destination_city",
        "route", "airline_class", "route_class", "airline_route",
        "route_stops", "departure_time", "arrival_time",
    ]

    numerical = [
        "flight_class_num", "stops_num", "dep_time_num", "arr_time_num",
        "duration", "days_left", "min_route_dur", "layover_est",
        "is_direct", "is_last_minute", "is_urgent", "is_early_bird",
        "days_left_inv", "days_left_log", "duration_log",
        "duration_per_stop", "speed_proxy",
        "flight_freq", "airline_freq", "route_freq", "airline_route_freq",
    ]

    return a, b, categorical, numerical

In [5]:
def train_forests(training, application, feature_function=original_features):
    # This retains the original feature-only preprocessing protocol.
    A, B, categories, numbers = feature_function(training, application)
    y = training['price'].to_numpy()
    y_log = np.log1p(y)
    oof_rf = np.zeros(len(training))
    oof_et = np.zeros(len(training))
    oof_blend = np.zeros(len(training))
    application_prediction = np.zeros(len(application))
    folds = KFold(5, shuffle=True, random_state=SEED)

    for fold, (fit_index, valid_index) in enumerate(folds.split(A)):
        encoder = TargetEncoder(target_type='continuous', cv=5,
                                smooth='auto', random_state=SEED)
        X_fit = np.column_stack([
            encoder.fit_transform(A.iloc[fit_index][categories], y_log[fit_index]),
            A.iloc[fit_index][numbers].to_numpy(),
        ])
        X_valid = np.column_stack([
            encoder.transform(A.iloc[valid_index][categories]),
            A.iloc[valid_index][numbers].to_numpy(),
        ])
        X_apply = np.column_stack([
            encoder.transform(B[categories]), B[numbers].to_numpy(),
        ])
        assert all(np.isfinite(X).all() for X in [X_fit, X_valid, X_apply])
        settings = dict(n_estimators=180, max_depth=25, min_samples_split=6,
                        min_samples_leaf=2, max_features=0.75,
                        random_state=SEED + fold, n_jobs=-1)
        rf = RandomForestRegressor(**settings)
        et = ExtraTreesRegressor(**settings)
        rf.fit(X_fit, y_log[fit_index])
        et.fit(X_fit, y_log[fit_index])
        rf_log = rf.predict(X_valid)
        et_log = et.predict(X_valid)
        oof_rf[valid_index] = np.expm1(rf_log)
        oof_et[valid_index] = np.expm1(et_log)
        oof_blend[valid_index] = np.expm1((rf_log + et_log) / 2)
        application_prediction += np.expm1(
            (rf.predict(X_apply) + et.predict(X_apply)) / 2
        ) / 5
        rmse = score(y[valid_index], oof_blend[valid_index])['RMSE']
        print(f'Fold {fold + 1}: RMSE={rmse:,.2f}')

    predictions = {'RF': oof_rf, 'ET': oof_et, 'RF + ET': oof_blend}
    results = pd.DataFrame([
        {'model': name, **score(y, prediction)}
        for name, prediction in predictions.items()
    ])
    application_prediction = np.clip(application_prediction, y.min(), y.max())
    return results, oof_blend, application_prediction

## 3. Define the LR model
Same segmentation, Ridge variants, fare statistics, and nearby-day stack as the simplified LR notebook. The grouped fare statistics have the same segment-wide fallback limitation.

In [6]:
def make_features(df, ref):
    X = df.copy()
    X['code'] = X.flight.str.split().str[0]
    X['dur_missing'] = X.duration.isna().astype(int)
    key = ['source_city', 'destination_city', 'stops']
    med = ref.groupby(key).duration.median().rename('dur_grp').reset_index()
    X = X.merge(med, on=key, how='left')
    X['duration'] = X.duration.fillna(X.dur_grp).fillna(ref.duration.median())
    X['dl_log'] = np.log1p(X.days_left)
    X['last3'] = (X.days_left <= 3).astype(int)
    return X
CATS = ['code', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city']
NUMS = ['duration', 'days_left', 'dl_log', 'last3', 'dur_missing']

In [7]:
from collections import defaultdict
DOMESTIC = {'6E', 'AI', 'IX'}
BASE_ALPHA = {'Economy': 100, 'BizDom': 500, 'BizInt': 500}
LEVELS = ['k_it', 'k_fl_r', 'k_fl', 'k_car_r']
STAT_NAMES = ['loo_med', 'loo_q25', 'loo_min', 'loo_mean']

def add_keys(X):
    dl = X.days_left.astype(str)
    X['k_it'] = X.flight + '|' + X.source_city + '|' + X.destination_city + '|' + X.departure_time + '|' + X.arrival_time + '|' + X.stops + '|' + X.flight_class
    X['k_fl_r'] = X.flight + '|' + X.source_city + '|' + X.destination_city + '|' + X.flight_class
    X['k_fl'] = X.flight + '|' + X.flight_class
    X['k_car_r'] = X.code + '|' + X.source_city + '|' + X.destination_city + '|' + X.stops + '|' + X.flight_class
    X['k_cs'] = X.code + '|' + X.stops + '|' + X.flight_class + '|' + dl
    X['k_f_dl'] = X.flight + '|' + X.flight_class + '|' + dl
    X['k_cr'] = X.code + '|' + X.source_city + '|' + X.destination_city + '|' + X.stops + '|' + X.flight_class + '|' + dl
    X['k_fr'] = X.flight + '|' + X.source_city + '|' + X.destination_city + '|' + X.flight_class + '|' + dl
    X['k_full'] = X.k_it + '|' + dl
    return X

def price_stats(A, y_a, B, leave_one_out):
    tables = []
    for lv in LEVELS:
        t = defaultdict(list)
        for i, (k, p) in enumerate(zip(A[lv].values, y_a)):
            t[k].append((i, p))
        tables.append(t)
    out = np.zeros((len(B), 4))
    for r in range(len(B)):
        for lv, t in zip(LEVELS, tables):
            items = t.get(B[lv].values[r], [])
            if leave_one_out:
                items = [(i, p) for i, p in items if i != r]
            if items:
                v = np.array([p for _, p in items])
                out[r] = (np.median(v), np.percentile(v, 25), v.min(), v.mean())
                break
        else:
            out[r] = (np.median(y_a), np.percentile(y_a, 25), y_a.min(), y_a.mean())
    return out
S_BASE = [('flight', 5), ('k_f_dl', 2), ('k_cr', 3), ('k_fr', 1), ('k_full', 0.1)]
VARIANTS = [dict(stats=['loo_med', 'loo_q25'], mult=5, stages=S_BASE), dict(stats=STAT_NAMES, mult=3, stages=S_BASE), dict(stats=['loo_med', 'loo_q25'], mult=10, stages=[('flight', 5), ('k_cs', 3)] + S_BASE[1:]), dict(stats=[], mult=1, stages=S_BASE)]

def segment(X):
    return np.where(X.flight_class == 'Economy', 'Economy', np.where(X.code.isin(DOMESTIC), 'BizDom', 'BizInt'))

def fit_predict(train, apply_to):
    Xtr = add_keys(make_features(train, train))
    Xap = add_keys(make_features(apply_to, train))
    str_, sap = (segment(Xtr), segment(Xap))
    pred = np.zeros(len(apply_to))
    for seg in np.unique(str_):
        itr, iap = (str_ == seg, sap == seg)
        if iap.sum() == 0:
            continue
        A, B = (Xtr[itr].reset_index(drop=True), Xap[iap].reset_index(drop=True))
        y = train.price.values[itr]
        sa, sb = (price_stats(A, y, A, True), price_stats(A, y, B, False))
        for j, nm in enumerate(STAT_NAMES):
            A[nm], B[nm] = (sa[:, j], sb[:, j])
        preds = []
        for v in VARIANTS:
            pre = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATS), ('n', StandardScaler(), NUMS + v['stats'])])
            stage1 = Pipeline([('pre', pre), ('poly', PolynomialFeatures(2, interaction_only=True, include_bias=False)), ('scale', StandardScaler()), ('ridge', Ridge(alpha=BASE_ALPHA[seg] * v['mult']))]).fit(A, y)
            p, resid = (stage1.predict(B), y - stage1.predict(A))
            for col, alpha in v['stages']:
                m = Pipeline([('oh', ColumnTransformer([('k', OneHotEncoder(handle_unknown='ignore'), [col])])), ('ridge', Ridge(alpha=alpha))]).fit(A, resid)
                p = p + m.predict(B)
                resid = resid - m.predict(A)
            preds.append(np.clip(p, y.min(), y.max() * 1.1))
        pred[np.where(iap)[0]] = np.mean(preds, axis=0)
    return pred

In [8]:
from sklearn.model_selection import KFold
from scipy.optimize import nnls
TAU = 5.0

def kernel_pred(Atr, y, Bap, key):
    tab = defaultdict(list)
    for k, dl, p in zip(Atr[key].values, Atr.days_left.values, y):
        tab[k].append((dl, p))
    tab = {k: np.array(v) for k, v in tab.items()}
    out = np.full(len(Bap), np.nan)
    for r, (k, dl) in enumerate(zip(Bap[key].values, Bap.days_left.values)):
        arr = tab.get(k)
        if arr is not None:
            w = np.exp(-np.abs(arr[:, 0] - dl) / TAU)
            out[r] = (w * arr[:, 1]).sum() / w.sum()
    return out

def stack_inputs(train, apply_to):
    P = fit_predict(train, apply_to)
    Xtr = add_keys(make_features(train, train))
    Xap = add_keys(make_features(apply_to, train))
    Ki = kernel_pred(Xtr, train.price.values, Xap, 'k_it')
    Kc = kernel_pred(Xtr, train.price.values, Xap, 'k_car_r')
    peer = ~np.isnan(Ki)
    Kc = np.where(np.isnan(Kc), P, Kc)
    Ki = np.where(np.isnan(Ki), Kc, Ki)
    return dict(X=np.column_stack([P, Ki, Kc]), seg=segment(Xap), peer=peer)
GROUPS = [(s, p) for s in ['Economy', 'BizDom', 'BizInt'] for p in (True, False)]

def oof_inputs(df, n_splits=5, seed=42):
    n = len(df)
    X = np.zeros((n, 3))
    seg = np.empty(n, dtype=object)
    peer = np.zeros(n, bool)
    fold = np.zeros(n, int)
    for f, (tr, va) in enumerate(KFold(n_splits, shuffle=True, random_state=seed).split(df)):
        r = stack_inputs(df.iloc[tr], df.iloc[va])
        X[va], seg[va], peer[va], fold[va] = (r['X'], r['seg'], r['peer'], f)
    return dict(X=X, seg=seg, peer=peer, fold=fold)

def fit_weights(inp, y, mask=None):
    mask = np.ones(len(y), bool) if mask is None else mask
    W = {}
    for s, p in GROUPS:
        m = mask & (inp['seg'] == s) & (inp['peer'] == p)
        W[s, p] = nnls(inp['X'][m], y[m])[0] if m.sum() >= 30 else np.array([1.0, 0.0, 0.0])
    return W

def apply_weights(inp, W):
    out = np.zeros(len(inp['X']))
    for s, p in GROUPS:
        m = (inp['seg'] == s) & (inp['peer'] == p)
        out[m] = inp['X'][m] @ W[s, p]
    return out

In [9]:
def predict_lr(training, application):
    # Learn combination weights from training-only OOF predictions.
    training = training.reset_index(drop=True)
    application = application.reset_index(drop=True)
    inputs = oof_inputs(training)
    weights = fit_weights(inputs, training['price'].to_numpy())
    return apply_weights(stack_inputs(training, application), weights)

## 4. Train on 60%, calibrate on 20%, evaluate on 20%
Calibration rows learn ensemble weights. Evaluation labels are not passed to either base model or used to fit combination weights. Feature-only forest lookups still use application rows, following the original preprocessing protocol.

In [10]:
fit_rows, remaining = train_test_split(train, test_size=0.4, random_state=SEED)
calibration, evaluation = train_test_split(remaining, test_size=0.5, random_state=43)
fit_rows = fit_rows.reset_index(drop=True)
calibration = calibration.reset_index(drop=True)
evaluation = evaluation.reset_index(drop=True)
application = pd.concat([calibration.drop(columns='price'), evaluation.drop(columns='price')], ignore_index=True)
print('Train / calibration / evaluation:', len(fit_rows), len(calibration), len(evaluation))
lr_prediction = predict_lr(fit_rows, application)
_, _, tree_prediction = train_forests(fit_rows, application)
predictions = np.column_stack([lr_prediction, tree_prediction])
P_cal = predictions[:len(calibration)]
P_eval = predictions[len(calibration):]

Train / calibration / evaluation: 9353 3118 3118


c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0004227853985779367.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.00083519544039359.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.00011393136075374867.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0099364540694491.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.

Fold 1: RMSE=12,184.06


c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Fold 2: RMSE=9,754.52


...

Fold 3: RMSE=11,674.83


c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Fold 4: RMSE=9,380.72


c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Fold 5: RMSE=11,192.66


## 5. Compare ensemble methods
Arithmetic averaging combines prices directly. Geometric averaging uses the square root of their product. Weighted averaging learns one LR weight; segment averaging learns separate weights. NNLS requires nonnegative coefficients; Ridge learns regularized coefficients and an intercept. Internal `dt_pred` variable names below refer only to RF + Extra Trees in this notebook.

In [11]:
def get_segment(rows):
    prefix = rows['flight'].str.split().str[0]
    return np.where(rows['flight_class'].eq('Economy'), 'Economy', np.where(prefix.isin({'6E', 'AI', 'IX'}), 'BizDom', 'BizInt'))

def optimal_lr_weight(predictions, actual):
    difference = predictions[:, 0] - predictions[:, 1]
    denominator = difference @ difference
    if denominator <= 1e-12:
        return 0.5
    weight = difference @ (actual - predictions[:, 1]) / denominator
    return float(np.clip(weight, 0.0, 1.0))

def fit_ensembles(predictions, actual, rows):
    global_weight = optimal_lr_weight(predictions, actual)
    nnls_weights = nnls(predictions, actual)[0]
    ridge = Ridge(alpha=10.0)
    ridge.fit(predictions / 1000.0, actual)
    segments = get_segment(rows)
    segment_weights = {}
    for segment in np.unique(segments):
        mask = segments == segment
        segment_weights[segment] = optimal_lr_weight(predictions[mask], actual[mask]) if mask.sum() >= 100 else global_weight
    return {'global_weight': global_weight, 'nnls_weights': nnls_weights, 'ridge': ridge, 'segment_weights': segment_weights}

def apply_ensembles(predictions, rows, fitted):
    lr_pred = predictions[:, 0]
    dt_pred = predictions[:, 1]
    weight = fitted['global_weight']
    segment_weights = np.array([fitted['segment_weights'].get(segment, weight) for segment in get_segment(rows)])
    return {'LR only': lr_pred, 'RF + ET only': dt_pred, 'Arithmetic 50/50': 0.5 * lr_pred + 0.5 * dt_pred, 'Geometric 50/50': np.sqrt(np.maximum(lr_pred, 0) * np.maximum(dt_pred, 0)), 'Weighted average': weight * lr_pred + (1 - weight) * dt_pred, 'NNLS stacking': predictions @ fitted['nnls_weights'], 'Ridge stacking': fitted['ridge'].predict(predictions / 1000.0), 'Segment-weighted average': segment_weights * lr_pred + (1 - segment_weights) * dt_pred}

In [12]:
fitted = fit_ensembles(P_cal, calibration['price'].to_numpy(), calibration)
evaluation_predictions = apply_ensembles(P_eval, evaluation, fitted)
results = pd.DataFrame([
    {'method': name, **score(evaluation['price'], prediction)}
    for name, prediction in evaluation_predictions.items()
]).sort_values('RMSE').reset_index(drop=True)
display(results.round(4))
print('Calibration LR weight:', fitted['global_weight'])

,method,RMSE,MAE,R2
0,Arithmetic 50/50,9532.9230,3207.8193,0.9355
1,Geometric 50/50,9537.6565,3203.7948,0.9354
2,Weighted average,9608.1868,3159.4860,0.9345
3,NNLS stacking,9639.0888,3116.0775,0.9341
4,Segment-weighted average,9640.5957,3169.3990,0.9340
5,Ridge stacking,9644.4166,3182.1596,0.9340
6,LR only,9682.5590,3447.2779,0.9335
7,RF + ET only,9888.5784,3175.7278,0.9306


Calibration LR weight: 0.31278329347134776


## 6. Train on all rows and export the chosen method
The default is the original 50/50 geometric rule. Review the new evaluation results before submitting; this code does not claim it is still the best method. Set `METHOD` to another exact name from the comparison table if desired. Learned combination weights remain the ones fitted on calibration data.

In [13]:
METHOD = 'Geometric 50/50'
lr_test = predict_lr(train, test)
_, _, tree_test = train_forests(train, test)
test_options = apply_ensembles(np.column_stack([lr_test, tree_test]), test, fitted)
submission = write_submission(test, test_options[METHOD], 'submission_ensemble_simple.csv')
display(submission.head())

c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.002854045924549671.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.005734600737340062.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0007078828654118859.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0031487061338419066.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh

Fold 1: RMSE=9,389.72


c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Fold 2: RMSE=10,229.41


c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


...

c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Fold 4: RMSE=10,914.06


c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Fold 5: RMSE=9,847.51
Saved: data\submission_ensemble_simple.csv


,id,price
0,1,8170.926362
1,2,7350.168187
2,3,29299.964417
3,4,32430.217172
4,5,4425.088527
